In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from picpy import LinearRegression, LogisticRegression
from picpy.penalties import L1Penalty, SCADPenalty, MCPPenalty
from picpy.calibration import BinomialCalibrator, GaussianCalibrator
from picpy.selection import StabilitySelection, RecursiveSelection
import cvxpy as cp

In [3]:
def standardize(X: np.ndarray) -> np.ndarray:
    return (X - X.mean(axis=0, keepdims=True)) / X.std(axis=0, ddof=0, keepdims=True)

def linear_data(n: int, p: int, s: int):
    X = standardize(np.random.randn(n, p))
    beta = np.zeros(p)
    true_features = np.random.permutation(p)[:s]
    beta[true_features] = 3.0
    y = X @ beta + np.random.randn(n).astype(np.float32)
    return X, y, true_features.tolist()

def logistic_data(n: int, p: int, s: int):
    X = standardize(np.random.randn(n, p))
    beta = np.zeros(p)
    true_features = np.random.permutation(p)[:s]
    beta[true_features] = 3.0
    prob = 1.0 / (1.0 + np.exp(-(X @ beta)))
    y = np.random.binomial(1, prob).astype(float)
    return X, y, true_features.tolist()

In [4]:
n, p, s = 150, 100, 13
X, y, true = linear_data(n, p, s)

# Standardisation identique à celle du modèle (centrage, écart-type biaisé)
mean, std = X.mean(axis=0), X.std(axis=0, ddof=0)
X = (X - mean) / np.where(std == 0, 1.0, std)

LAM = GaussianCalibrator(n_simu=2000, alpha=0.05).compute(X)

# 1) Modèle, avec ce λ imposé (aucune recalibration interne)
model = LinearRegression(lambda_=LAM, penalty=L1Penalty())
model.fit(X, y)
assert float(model.lambda_) == LAM

# 2) cvxpy, avec le même λ et le même X standardisé
beta, bias = cp.Variable(p), cp.Variable()
objective = cp.norm2(y - X @ beta - bias) / np.sqrt(n) + LAM * cp.norm1(beta)
problem = cp.Problem(cp.Minimize(objective))
problem.solve(solver=cp.CLARABEL, tol_gap_abs=1e-10, tol_gap_rel=1e-10, tol_feas=1e-10)

# Comparaison
beta_mod = model.coef_
obj_mod = np.linalg.norm(y-model.predict(X)) / np.sqrt(n) + model.penalty_value
support_cvx = np.flatnonzero(np.abs(beta.value) > 1e-6)

print("λ (les deux)       :", LAM)
print("statut cvxpy       :", problem.status)
print("support cvxpy      :", sorted(support_cvx.tolist()))
print("support modèle     :", sorted(model.selected_features_.tolist()))
print("vraies variables   :", sorted(true))
print("max |β_mod − β_cvx|:", np.abs(beta_mod - beta.value).max())
print("|b_mod − b_cvx|    :", abs(model.intercept_ - bias.value))
print("objectif cvxpy     :", problem.value)
print("objectif modèle    :", obj_mod)

λ (les deux)       : 0.28025091498715665
statut cvxpy       : optimal
support cvxpy      : [5, 7, 23, 37, 42, 43, 45, 60, 67, 77, 91, 93, 99]
support modèle     : [5, 7, 23, 37, 42, 43, 45, 60, 67, 77, 91, 93, 99]
vraies variables   : [5, 7, 23, 37, 42, 43, 45, 60, 67, 77, 91, 93, 99]
max |β_mod − β_cvx|: 9.482576873787174e-05
|b_mod − b_cvx|    : 4.996003610813204e-16
objectif cvxpy     : 11.032435443398395
objectif modèle    : 11.032435442429087


In [5]:
import cvxpy as cp

n, p, s = 500, 100, 4
X, y, true = logistic_data(n, p, s)

LAM = BinomialCalibrator(n_simu=2000, alpha=0.05).compute(X)

# 1) Modèle, avec ce λ imposé
model = LogisticRegression(lambda_=LAM, penalty=L1Penalty())
model.fit(X, y)

# 2) cvxpy : avec µ = σ(η), sqrt((1-µ)/µ) = exp(-η/2)
#    ℓ = (2/n) Σ [ y exp(-η/2) + (1-y) exp(η/2) ]
beta, bias = cp.Variable(p), cp.Variable()
eta = X @ beta + bias
loss = (2.0 / n) * cp.sum(
    cp.multiply(y, cp.exp(-eta / 2)) + cp.multiply(1.0 - y, cp.exp(eta / 2))
)
problem = cp.Problem(cp.Minimize(loss + LAM * cp.norm1(beta)))
problem.solve(solver=cp.CLARABEL, tol_gap_abs=1e-10, tol_gap_rel=1e-10, tol_feas=1e-10)

# Comparaison
eta_mod = X @ model.coef_ + model.intercept_
obj_mod = model.family.evaluate(y, eta_mod) + model.penalty_value
support_cvx = np.flatnonzero(np.abs(beta.value) > 1e-6)

print("λ (les deux)       :", LAM)
print("statut cvxpy       :", problem.status)
print("support cvxpy      :", sorted(support_cvx.tolist()))
print("support modèle     :", sorted(model.selected_features_.tolist()))
print("vraies variables   :", sorted(true))
print("max |β_mod − β_cvx|:", np.abs(model.coef_ - beta.value).max())
print("|b_mod − b_cvx|    :", abs(model.intercept_ - bias.value))
print("objectif cvxpy     :", problem.value)
print("objectif modèle    :", obj_mod)


λ (les deux)       : 0.156073448962723
statut cvxpy       : optimal
support cvxpy      : [23, 61, 63, 91]
support modèle     : [23, 61, 63, 91]
vraies variables   : [23, 61, 63, 91]
max |β_mod − β_cvx|: 6.39182531902982e-07
|b_mod − b_cvx|    : 4.286551391619042e-08
objectif cvxpy     : 1.7175844892174177
objectif modèle    : 1.7175844892143304


In [6]:
n, p, s = 100, 200, 4
X, y, true = linear_data(n, p, s)

LAM = GaussianCalibrator(n_simu=2000, alpha=0.05).compute(X)

model = LinearRegression(lambda_=LAM, penalty=L1Penalty())
model.fit(X, y)
model.selected_features_

array([ 61,  92, 128, 164])

In [9]:
calibrator = GaussianCalibrator()
stab = StabilitySelection(model, calibrator=calibrator, n_resamples=500, sample_fraction=0.8,
                   stratify=True, random_state=40, n_jobs=-1, progress=True)
stab.fit(X, y)
stab.stability_

Stability Selection: 100%|██████████| 500/500 [00:01<00:00, 356.00it/s]


{'mean_jaccard': 0.9960106880427522,
 'std_jaccard': 0.028014245060169606,
 'nogueira': 0.9974592912239856,
 'mean_n_selected': 4.01,
 'std_n_selected': 0.09949874371066199,
 'n_selected_distribution': {4: 495, 5: 5}}

In [10]:
calibrator = GaussianCalibrator()
recursive = RecursiveSelection(
    model=model,
    calibrator=calibrator,
    max_iter=20,
    progress=True,
)

recursive.fit(X, y)

Recursive Selection:   5%|▌         | 1/20 [00:00<00:13,  1.38it/s, remaining=196, selected=4]


RecursiveSelection(model=LinearRegression(lambda_=0.3584246107898948, penalty=L1Penalty, fit_intercept=True, refit=False, tol=1e-06, max_iter=500, verbosity=0, log_interval=5), calibrator=GaussianCalibrator(alpha=0.05, method='mc_exact', n_simu=2000, batch_size=None, fit_intercept=True, random_state=None), max_iter=20, min_features=1, min_selected=1, random_state=42, progress=True)

In [11]:
display(recursive.summary())

,iteration,lambda,n_selected,n_remaining,converged,evaluation,mse,rmse,r2
0,1,0.358971,4,196,True,train,1.027268,1.013543,0.971203
